## Prepare questions from PPMI for testing GRPO model

Created by: Sahana Kowshik

Date: 11/01/2025

In [1]:
import pandas as pd
import json
import random

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/ppmi"
test = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/test_summary.csv")

In [3]:
print(test.iloc[0]['visit_summary'])

The subject is a 75-year-old man who identifies as a man (or trans man) and is heterosexual. He was enrolled at the age of 74.98 and has six years of education. He is White, not of Hispanic/Latino ethnicity, and does not identify with Ashkenazi Jewish, African Berber, or Basque descent. He is right-handed. There is no family history of Parkinson's disease, as indicated by both the three-category and binary classifications. The subject is currently receiving dopaminergic medication for Parkinson's disease, with a total Levodopa Equivalent Daily Dose of 450 mg. His body mass index is 23.99, which falls within the normal range, and there is no indication of significant changes in systolic or diastolic blood pressure. According to the Unified Parkinson's Disease Rating Scale (UPDRS), he is classified in Stage 1 for both OFF and ON states. His cognitive impairment is described as slight, and he does not exhibit hallucinations, psychosis, depressed mood, anxious mood, apathy, or fatigue. Fea

In [4]:
len(test)

3315

In [5]:
test.head()

,SITE,PATNO,COHORT,subgroup,enroll_phase,HIQ_RBD,study_status,NSD_Status,NSD_STAGE,PRIMDIAG,...,Stage_subpark,Stage_PDTreat,Stage_S,Stage_D,Stage_G,patient_summary,diag_summary,brain_findings_summary,visit_summary_prompt,visit_summary
0,67,214241,PPMI,Sporadic PD,2,NaN,Active,1.0,3,1.0,...,25.0,1.0,1.0,1.0,NaN,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""Idiopathic ...",NaN,<|im_start|>user\nYou will receive patient dat...,The subject is a 75-year-old man who identifie...
1,40,153045,PPMI,Hyposmia,2,1.0,Active,NaN,NaN,25.0,...,NaN,NaN,NaN,NaN,NaN,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""Prodromal S...",NaN,<|im_start|>user\nYou will receive patient dat...,The subject is a 64.8-year-old male with a sel...
2,28,3457,PPMI,Healthy Control,1,NaN,Withdrawn,NaN,NaN,17.0,...,NaN,NaN,NaN,NaN,NaN,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""No PD nor o...",Hippocampus region has moderate atrophy; Media...,<|im_start|>user\nYou will receive patient dat...,The subject is a 68-year-old female who was en...
3,39,224562,PPMI,RBD,2,NaN,Active,NaN,NaN,25.0,...,NaN,NaN,NaN,NaN,NaN,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""Prodromal S...",Inferior Ventricles region has moderate enlarg...,<|im_start|>user\nYou will receive patient dat...,The subject is a 79-year-old male who enrolled...
4,21,58026,PPMI,LRRK2,1,NaN,Active,NaN,NaN,17.0,...,NaN,NaN,NaN,NaN,NaN,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""No PD nor o...",NaN,<|im_start|>user\nYou will receive patient dat...,The patient is a 71.59-year-old right-handed f...


In [6]:
test['cogstate'].isna().sum()

0

In [7]:
test['PRIMDIAG'].isna().sum()

0

In [8]:
test['cogstate'].value_counts()

cogstate
1.0    2757
2.0     495
3.0      63
Name: count, dtype: int64

In [9]:
columns = ['ID', 'patient_summary', 'diag_summary', 'visit_summary', 'question', 'options', 'ground_truth', 'ground_truth_text', 'COHORT']

# Add cognitive status question


In [10]:
import random, string

# Question variants for cognitive status identification
cog_question_variants = [
    "Using the information provided, determine the patient's cognitive status.",
    "Identify the patient's cognitive status based on the given information.",
    "Based on the provided clinical details, classify the patient's cognitive status.",
    "From the information available, determine the correct cognitive status for the patient.",
    "Analyze the patient's information and identify their cognitive status."
]

# Original answer texts
COG_OPTION_TEXTS = [
    "Normal Cognition (NC)",
    "Mild Cognitive Impairment (MCI)",
    "Dementia (DE)"
]

# Mapping from NACCUDSD to the correct label text
COG_ANSWER_MAP = {
    1: COG_OPTION_TEXTS[0],
    2: COG_OPTION_TEXTS[1],
    3: COG_OPTION_TEXTS[2],
}

def add_cog_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Random question phrasing
    row['question'] = rng.choice(cog_question_variants)

    # 2️⃣ Shuffle the answer options
    shuffled = COG_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Determine correct answer letter
    correct_text = COG_ANSWER_MAP.get(row['cogstate'], None)
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # fallback for unrecognized codes
    
    row['ground_truth_text'] = correct_text

    return row


In [11]:
# Apply the function to the MCI training set
test_cog = test.apply(add_cog_question, axis=1)

In [12]:
test_cog['ground_truth_text'].value_counts()

ground_truth_text
Normal Cognition (NC)              2757
Mild Cognitive Impairment (MCI)     495
Dementia (DE)                        63
Name: count, dtype: int64

In [13]:
test_cog["ID"] = test_cog["PATNO"]
test_cog = test_cog[columns]
print(len(test_cog))
test_cog = test_cog.dropna(how='any').reset_index(drop=True)
print(len(test_cog))

3315
3315


/scratch/ipykernel_3304702/1816999620.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_cog["ID"] = test_cog["PATNO"]


In [14]:
test_cog['Q_TYPE'] = "Cognitive status"

In [15]:
test_cog.head()

,ID,patient_summary,diag_summary,visit_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,214241,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""Idiopathic ...",The subject is a 75-year-old man who identifie...,"From the information available, determine the ...",A. Dementia (DE)\nB. Normal Cognition (NC)\nC....,C,Mild Cognitive Impairment (MCI),PPMI,Cognitive status
1,153045,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""Prodromal S...",The subject is a 64.8-year-old male with a sel...,Identify the patient's cognitive status based ...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,B,Normal Cognition (NC),PPMI,Cognitive status
2,3457,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""No PD nor o...",The subject is a 68-year-old female who was en...,"Using the information provided, determine the ...",A. Mild Cognitive Impairment (MCI)\nB. Dementi...,C,Normal Cognition (NC),PPMI,Cognitive status
3,224562,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""Prodromal S...",The subject is a 79-year-old male who enrolled...,Identify the patient's cognitive status based ...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,B,Normal Cognition (NC),PPMI,Cognitive status
4,58026,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""No PD nor o...",The patient is a 71.59-year-old right-handed f...,Identify the patient's cognitive status based ...,A. Dementia (DE)\nB. Normal Cognition (NC)\nC....,B,Normal Cognition (NC),PPMI,Cognitive status


In [16]:
test_cog.to_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_cog.csv", index=False)

# Add PD question

In [17]:
test['PRIMDIAG'].value_counts()

PRIMDIAG
1.0     1373
17.0     878
25.0     867
97.0      91
7.0       40
23.0      19
24.0      19
5.0        7
15.0       6
11.0       5
2.0        4
9.0        2
18.0       2
4.0        1
16.0       1
Name: count, dtype: int64

In [18]:
test_pd = test[test['cogstate'].isin([2, 3])].copy().reset_index(drop=True)

In [19]:
test_pd['PRIMDIAG'].value_counts()

PRIMDIAG
1.0     357
25.0    107
17.0     48
97.0     16
23.0      7
5.0       7
7.0       5
2.0       4
24.0      3
11.0      2
4.0       1
15.0      1
Name: count, dtype: int64

In [20]:
import random, string

etiology_question_variants = [
    "Identify the primary etiologic diagnosis of the patient using the information provided, if applicable.",
    "Identify the primary etiology of the patient's cognitive impairment using the information provided, if applicable.",
    "Based on the clinical details, determine the main cause of the patient's cognitive impairment, if applicable.",
    "Using the information provided, identify the primary cause of the patient's cognitive decline, if applicable.",
    "Determine the primary etiology underlying the patient's cognitive impairment based on the provided information, if applicable."
]


# Raw etiology answer texts in original order
ETIOLOGY_OPTION_TEXTS = [
    "Idiopathic Parkinson's Disease",
    "Prodromal Parkinson's Disease",
    "No PD nor other neurological disorder",
    "Other neurological disorder(s)"
]

# Mapping of FTLD code to correct answer text
ETIOLOGY_ANSWER_MAP = {
    1: ETIOLOGY_OPTION_TEXTS[0],
    23: ETIOLOGY_OPTION_TEXTS[1],
    24: ETIOLOGY_OPTION_TEXTS[1],
    25: ETIOLOGY_OPTION_TEXTS[1],
    17: ETIOLOGY_OPTION_TEXTS[2],
    2: ETIOLOGY_OPTION_TEXTS[3],
    4: ETIOLOGY_OPTION_TEXTS[3],
    5: ETIOLOGY_OPTION_TEXTS[3],
    7: ETIOLOGY_OPTION_TEXTS[3],
    11: ETIOLOGY_OPTION_TEXTS[3],
    15: ETIOLOGY_OPTION_TEXTS[3],
    97: ETIOLOGY_OPTION_TEXTS[3],
}

def add_etpr_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Randomly select a question variant
    row['question'] = rng.choice(etiology_question_variants)

    # 2️⃣ Shuffle options and assign fresh letters
    shuffled = ETIOLOGY_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Get correct text from PD code (default to "Not applicable" if unknown)
    correct_text = ETIOLOGY_ANSWER_MAP.get(row['PRIMDIAG'])

    if correct_text is None:
        raise ValueError(f"Invalid PD value: {row['PRIMDIAG']}")


    # 4️⃣ Get ground truth letter from shuffled list
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # shouldn't happen unless text mismatch
        
    row['ground_truth_text'] = correct_text

    return row


In [21]:
# Apply the function to the MCI training set
test_pd = test_pd.apply(add_etpr_question, axis=1)

In [22]:
test_pd[test_pd['ground_truth'].isna()]

,SITE,PATNO,COHORT,subgroup,enroll_phase,HIQ_RBD,study_status,NSD_Status,NSD_STAGE,PRIMDIAG,...,Stage_G,patient_summary,diag_summary,brain_findings_summary,visit_summary_prompt,visit_summary,question,options,ground_truth,ground_truth_text


In [23]:
test_pd["ID"] = test_pd["PATNO"]
test_pd = test_pd[columns]
print(len(test_pd))
test_pd = test_pd.dropna(how='any').reset_index(drop=True)
print(len(test_pd))

558
558


/scratch/ipykernel_3304702/3739126350.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_pd["ID"] = test_pd["PATNO"]


In [24]:
test_pd['ground_truth_text'].value_counts()

ground_truth_text
Idiopathic Parkinson's Disease           357
Prodromal Parkinson's Disease            117
No PD nor other neurological disorder     48
Other neurological disorder(s)            36
Name: count, dtype: int64

In [25]:
test_pd['Q_TYPE'] = "Primary etiology"

In [26]:
test_pd.head()

,ID,patient_summary,diag_summary,visit_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,214241,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""Idiopathic ...",The subject is a 75-year-old man who identifie...,"Using the information provided, identify the p...",A. No PD nor other neurological disorder\nB. P...,C,Idiopathic Parkinson's Disease,PPMI,Primary etiology
1,43082,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""Idiopathic ...",The subject is a 78.7-year-old male at enrollm...,Identify the primary etiology of the patient's...,A. No PD nor other neurological disorder\nB. O...,D,Idiopathic Parkinson's Disease,PPMI,Primary etiology
2,3654,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""Idiopathic ...",The subject is a 60.66-year-old right-handed f...,Identify the primary etiologic diagnosis of th...,A. No PD nor other neurological disorder\nB. P...,D,Idiopathic Parkinson's Disease,PPMI,Primary etiology
3,116231,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""Other neuro...",The subject is an 83-year-old right-handed fem...,Identify the primary etiology of the patient's...,A. Idiopathic Parkinson's Disease\nB. No PD no...,C,Other neurological disorder(s),PPMI,Primary etiology
4,102308,"{""Subject Demographics"": {""Age at Enrollment"":...","{""Most likely Primary Diagnosis"": ""Idiopathic ...",The subject is a 62.5-year-old male who was en...,Identify the primary etiology of the patient's...,A. Other neurological disorder(s)\nB. Prodroma...,C,Idiopathic Parkinson's Disease,PPMI,Primary etiology


In [27]:
test_pd.to_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_etpr.csv", index=False)

# Datscan

In [28]:
test['Stage_D'].value_counts()

Stage_D
1.0    1216
0.0      60
Name: count, dtype: int64

In [29]:
test_dat = test[~test['Stage_D'].isna()].copy().reset_index(drop=True)

In [30]:
json.loads(test_dat.iloc[1]['patient_summary'])

{'Subject Demographics': {'Age at Enrollment': 60.8630136986301,
  'Age at Visit': 60.8630136986301,
  'Sex at birth': 'Male',
  'Years of Education capped at 20': 20,
  'Race': 'White',
  'Indicator for Hispanic/Latino ethnicity': 'No',
  'Handedness': 'Right'},
 'Subject Family History': {'Family History of PD - 3 categories.': 'No Family w/PD',
  'Family History of PD - Binary': 'No Family w/PD'},
 'Subject Medications': {'Is the participant on dopaminergic medication or receiving deep brain stimulation for treating the symptoms of Parkinsons disease?': 'No',
  'Total Levodopa Equivalent Daily Dose': 0.0},
 'Physical': {'Body Mass Index': 25.4716077755434,
  'Indicator of whether systolic blood pressure change is greater or equal to 20 OR diastolic blood pressure change is greater than or equal to 10.': 'No'},
 "Unified Parkinson's Disease Rating Scale (UPDRS)": {'Hoehn & Yahr Stage (includes OFF and untreated scores)': 'Stage 2',
  'Hoehn & Yahr Stage (includes ON and untreated sco

In [31]:
import random, string, json

# Keys related to DATScan that should be removed from the patient summary
dat_keys_remove = [
    'DATSCAN'
]

dat_question_variants = [
    "Is there evidence of Lewy body disease in the brain based on the provided information?",
    "Does the provided information suggest the presence of Lewy body disease in the brain?",
    "Is the patient likely to have Lewy body pathology in the brain based on the available data?",
    "Based on the information given, is there evidence consistent with Lewy body disease in the brain?",
    "Does the information support the presence of underlying Lewy body pathology in the brain?"
]

# Raw answer texts
DAT_OPTION_TEXTS = ["Yes", "No"]

# Map DATSCAN value to answer text
DAT_ANSWER_MAP = {
    1: "Yes",  # DATScan positive
    0: "No",   # DATScan negative
}

def add_dat_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Randomly assign a question variant
    row['question'] = rng.choice(dat_question_variants)

    # 2️⃣ Remove sensitive DATScan key from patient summary
    json_file = json.loads(row['patient_summary'])
    for key in dat_keys_remove:
        if key in json_file.get('Imaging evidence', {}):
            json_file['Imaging evidence'].pop(key)
        if len(json_file['Imaging evidence']) == 0:
            json_file.pop('Imaging evidence')
    row['patient_summary'] = json.dumps(json_file, indent=4)

    # 3️⃣ Shuffle options
    shuffled = DAT_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row["options"] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 4️⃣ Determine correct letter based on DATSCAN value
    correct_text = DAT_ANSWER_MAP[row['Stage_D']]
    correct_letter = letters[shuffled.index(correct_text)]
    
    row["ground_truth"] = correct_letter
    row['ground_truth_text'] = correct_text

    return row


In [32]:
# Apply the function to the MCI training set
test_dat = test_dat.apply(add_dat_question, axis=1)

In [33]:
test_dat.iloc[1]['visit_summary']

"The subject is a 60.86-year-old right-handed male of White race and non-Hispanic/Latino ethnicity who has completed 20 years of education. There is no family history of Parkinson's disease. The participant is not currently taking dopaminergic medication or receiving deep brain stimulation for Parkinson's disease, and the total levodopa equivalent daily dose is 0.0. His body mass index is 25.47, and there is no indication of significant changes in systolic or diastolic blood pressure. \n\nOn the Unified Parkinson's Disease Rating Scale (UPDRS), the subject is classified at Hoehn & Yahr Stage 2 both on and off medication. The MDS-UPDRS Part I scores indicate normal cognitive function, absence of hallucinations and psychosis, and no signs of depression, anxiety, apathy, dopamine dysregulation syndrome, or fatigue. The MDS-UPDRS Part II score is 2, and the Part III score is 19 both on and off medication, with a total score of 23 in both conditions. The Geriatric Depression Scale (GDS) sco

In [34]:
json.loads(test_dat[test_dat['PATNO'] == 214241].iloc[0]['patient_summary'])

{'Subject Demographics': {'Age at Enrollment': 74.9780821917808,
  'Age at Visit': 75.9808219178082,
  'Sex at birth': 'Male',
  'Years of Education capped at 20': 6,
  'Race': 'White',
  'Indicator for Hispanic/Latino ethnicity': 'No',
  'Indicator for Ashkenazi Jewish descent': 'No',
  'Indicator for African Berber descent': 'No',
  'Indicator for Basque descent': 'No',
  'Handedness': 'Right',
  'Gender Identity': 'Man (or trans man)',
  'Sexual Orientation': 'Heterosexual'},
 'Subject Family History': {'Family History of PD - 3 categories.': 'No Family w/PD',
  'Family History of PD - Binary': 'No Family w/PD'},
 'Subject Medications': {'Is the participant on dopaminergic medication or receiving deep brain stimulation for treating the symptoms of Parkinsons disease?': 'Yes',
  'Total Levodopa Equivalent Daily Dose': 450.0},
 'Physical': {'Body Mass Index': 23.9994591671174,
  'Indicator of whether systolic blood pressure change is greater or equal to 20 OR diastolic blood pressure 

In [35]:
test_dat['ground_truth_text'].value_counts()

ground_truth_text
Yes    1216
No       60
Name: count, dtype: int64

In [36]:
test_dat["ID"] = test_dat["PATNO"]
test_dat = test_dat[columns]
print(len(test_dat))
test_dat = test_dat.dropna(how='any').reset_index(drop=True)
print(len(test_dat))

1276
1276


/scratch/ipykernel_3304702/788674593.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_dat["ID"] = test_dat["PATNO"]


In [37]:
test_dat['Q_TYPE'] = "Datscan"

In [38]:
test_dat.head()

,ID,patient_summary,diag_summary,visit_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,214241,"{\n ""Subject Demographics"": {\n ""Age...","{""Most likely Primary Diagnosis"": ""Idiopathic ...",The subject is a 75-year-old man who identifie...,"Based on the information given, is there evide...",A. Yes\nB. No,A,Yes,PPMI,Datscan
1,4066,"{\n ""Subject Demographics"": {\n ""Age...","{""Most likely Primary Diagnosis"": ""Other neuro...",The subject is a 60.86-year-old right-handed m...,Does the provided information suggest the pres...,A. No\nB. Yes,B,Yes,PPMI,Datscan
2,140568,"{\n ""Subject Demographics"": {\n ""Age...","{""Most likely Primary Diagnosis"": ""Idiopathic ...",The patient is a 60-year-old right-handed fema...,Is there evidence of Lewy body disease in the ...,A. No\nB. Yes,B,Yes,PPMI,Datscan
3,324862,"{\n ""Subject Demographics"": {\n ""Age...","{""Most likely Primary Diagnosis"": ""Idiopathic ...",The subject is a 56.9-year-old male who identi...,Does the provided information suggest the pres...,A. No\nB. Yes,B,Yes,PPMI,Datscan
4,190603,"{\n ""Subject Demographics"": {\n ""Age...","{""Most likely Primary Diagnosis"": ""Idiopathic ...",The subject is a 65-year-old male who enrolled...,Does the provided information suggest the pres...,A. Yes\nB. No,A,Yes,PPMI,Datscan


In [39]:
test_dat.to_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_dat_unmodified_summaries.csv", index=False)